# ML-03 -- Frame Your Lane as an ML Task

This notebook frames the content-decline prediction task for FlyRank: deciding which content pages an editorial team should prioritise for a refresh, using `data/raw/content_refresh_anonymized.csv` (30 000 rows, 44 columns, trailing-90-day metrics).

> Skills loaded: `framing-ml-problems/SKILL.md`, `flyrank/flyrank-data/SKILL.md`

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring -- which one, and why?*

This task is best framed as **ranking / scoring**.

Although the label `is_declining_label` is binary (the kind that naturally trains a classifier), the business goal is not a binary yes/no answer -- it is a **priority queue**. Editors have limited bandwidth; they can act on maybe 50-100 pages per week. A raw 'declining / not declining' bucket is useless without an order. The classifier's output probability becomes the score used to rank all 30 000 pages from highest to lowest decline risk, and the team works down that list.

In the framing-skill taxonomy:

| Question | Answer |
|---|---|
| Task type | Ranking / Scoring (classifier trained, score used to sort) |
| Target | `is_declining_label` = (`trend_direction == 'down'`) |
| Typical metric | Precision@K -- how many of the top-K scored pages truly need a refresh |

In [1]:
# No computation needed for this section -- task type is a framing decision.

## 2. Target or proxy

*What would you predict? Where does that label come from -- observed outcome or a defined rule?*

**Target:** `is_declining_label` -- a binary column (True / False) derived as `trend_direction == 'down'`. It indicates whether a content page's search traffic is in decline over the trailing 90-day window.

**Observed outcome, not a proxy.** `trend_direction` classifies the actual measured percentage change in traffic (`trend_pct`) that occurred in the real world. It is not an arbitrary business rule invented after the fact (e.g. 'flag if position > 10') -- it reflects a physical outcome: real impressions and clicks either fell ('down'), grew ('up'), or were flat.

**The leakage rule that follows from this:** because `is_declining_label` is derived directly from `trend_direction` and `trend_pct`, **both of those columns must be dropped from the feature set before any modelling**. Using them would make the model memorise the arithmetic formula, not learn the underlying pattern from independent signals.

The code below derives the label, checks its distribution, and confirms the leakage columns are dropped from the modelling dataframe.

In [2]:
import pandas as pd
import numpy as np
_CSV = r'C:/Users/Admin/Desktop/Krish/Projects/Flyrank ML/Machine Learning/data/raw/content_refresh_anonymized.csv'

raw = pd.read_csv(_CSV)

# Derive the label before dropping the source columns
raw['is_declining_label'] = (raw['trend_direction'] == 'down')

# Confirm leakage columns exist in raw
leakage_cols = ['trend_pct', 'trend_direction']
print('Leakage columns present in raw CSV:', [c for c in leakage_cols if c in raw.columns])

# Label distribution
print('\nis_declining_label value counts:')
print(raw['is_declining_label'].value_counts())
print(f'\nDecline rate: {raw["is_declining_label"].mean():.1%}')
print('\nNote: rows where trend_direction is NaN (new content, no history) are NOT labelled as declining.')
print(f'NaN trend_direction rows: {raw["trend_direction"].isna().sum()}')

Leakage columns present in raw CSV: ['trend_pct', 'trend_direction']

is_declining_label value counts:
is_declining_label
True     16262
False    13738
Name: count, dtype: int64

Decline rate: 54.2%

Note: rows where trend_direction is NaN (new content, no history) are NOT labelled as declining.
NaN trend_direction rows: 0


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@100** (or Precision@K where K = the team's weekly editing capacity).

**Why not accuracy?** If ~54% of pages are labelled declining (mostly 'down' rows), accuracy is misleading because it treats false positives and false negatives as equally bad -- but the business cost is asymmetric. Wasting an editor's hours on a healthy page (false positive) is a different cost from missing a declining page (false negative).

**Why Precision@100?** Editors can only act on a fixed number of pages per week. The only thing that matters is: of the 100 pages the model ranks highest, how many genuinely needed a refresh? A Precision@100 of 0.80 means 80 of the 100 pages the editor touches are real work -- 80% useful signal, 20% wasted effort. This directly maps to the business cost framing.

**Secondary metric: PR-AUC** (Precision-Recall Area Under Curve). Better than ROC-AUC for understanding model quality across all thresholds because it focuses entirely on the declining-page class without inflating from true negatives. Useful for comparing model variants.

The code below computes a naive baseline Precision@100 (sorting by lowest `ctr`) to give us a concrete floor to beat.

In [3]:
import pandas as pd
_CSV = r'C:/Users/Admin/Desktop/Krish/Projects/Flyrank ML/Machine Learning/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(_CSV)
df['is_declining_label'] = (df['trend_direction'] == 'down')

K = 100

# Naive baseline: rank by lowest ctr (most likely to be in trouble)
top_k = df.nsmallest(K, 'ctr')
baseline_precision = top_k['is_declining_label'].mean()

# Random baseline = overall decline rate
random_baseline = df['is_declining_label'].mean()

print(f'Naive Precision@{K} (lowest ctr): {baseline_precision:.2%}')
print(f'Random baseline (overall decline rate): {random_baseline:.2%}')
print(f'\nA useful model must beat the naive single-signal baseline of {baseline_precision:.2%}')

Naive Precision@100 (lowest ctr): 51.00%
Random baseline (overall decline rate): 54.21%

A useful model must beat the naive single-signal baseline of 51.00%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content page**, representing its aggregated performance over a trailing 90-day window. This is a cross-sectional snapshot (not a time series panel): there is exactly one row per content item, not one row per content item per day.

Gotchas applied before any modelling:
- `is_declining_label` derived from `trend_direction` then both **`trend_pct` and `trend_direction` dropped** (leakage -- they derive the label)
- `avg_position = 0` replaced with NaN then filled with sentinel `999` (0 means no data, not rank zero)
- `has_<col>` boolean flags added for any column with missingness (missingness correlates with `content_type`; a blind `fillna(0)` would inject a category signal)
- Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are x100 percentages -- `ctr = 0.76` means 0.76%, not 76%
- `content_id` and `client_id` are pseudonyms -- used only for grouping/splitting, never as features

In [4]:
import pandas as pd
import numpy as np
_CSV = r'C:/Users/Admin/Desktop/Krish/Projects/Flyrank ML/Machine Learning/data/raw/content_refresh_anonymized.csv'

# Load
df = pd.read_csv(_CSV)
print(f'Raw shape: {df.shape}  ({df.shape[0]:,} rows x {df.shape[1]} columns)')

# Derive label first, then drop source columns
df['is_declining_label'] = (df['trend_direction'] == 'down')
df = df.drop(columns=['trend_direction', 'trend_pct'], errors='ignore')

# Fix avg_position (0 = missing, not rank zero)
df['avg_position'] = df['avg_position'].replace(0, np.nan)

# Missingness flags before filling
for col in df.columns:
    if df[col].isnull().any():
        df[f'has_{col}'] = ~df[col].isnull()

# avg_position: sentinel 999 (clearly not a real rank)
df['avg_position'] = df['avg_position'].fillna(999)

# Other numeric: 0
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Categorical: 'missing'
string_cols = df.select_dtypes(exclude=[np.number]).columns
df[string_cols] = df[string_cols].fillna('missing')

print(f'Clean shape: {df.shape}')
print('Unit of analysis: one row = one content page (90-day snapshot)')
print()
df.info()
print()
df[['content_id', 'client_id', 'content_type', 'ctr',
    'avg_position', 'engagement_rate', 'word_count',
    'has_word_count', 'has_avg_position',
    'is_declining_label']].head()

Raw shape: (30000, 44)  (30,000 rows x 44 columns)
Clean shape: (30000, 56)
Unit of analysis: one row = one content page (90-day snapshot)

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 56 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  str    
 1   client_id               30000 non-null  str    
 2   search_volume           30000 non-null  float64
 3   competition             30000 non-null  float64
 4   competition_level       30000 non-null  str    
 5   cpc                     30000 non-null  float64
 6   content_type            30000 non-null  str    
 7   main_intent             30000 non-null  str    
 8   word_count              30000 non-null  float64
 9   char_count              30000 non-null  float64
 10  provider_used           30000 non-null  str    
 11  model_used              30000 non-null  str    
 12  impressions_90d

,content_id,client_id,content_type,ctr,avg_position,engagement_rate,word_count,has_word_count,has_avg_position,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,0.76,10.6,5.88,3221.0,True,True,True
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0.05,20.3,0.00,2481.0,True,True,True
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.09,36.5,0.00,3515.0,True,True,True
3,content_331d6c4de07b,client_19581e27de,keyword article,0.49,6.2,1.28,0.0,False,True,False
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.13,44.0,0.00,2803.0,True,True,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A hand-written rule for flagging declining pages might look like:

```
IF avg_position > 20 AND ctr < 0.5 THEN flag_for_refresh
```

This fails for several reasons:

1. **Single-signal fragility.** A drop in `avg_position` could reflect a Google algorithm test, a seasonal slowdown, or a broken tracking pixel -- not genuine content decay. The same drop combined with falling `engagement_rate` and low `scroll_rate` is a much more reliable decay signal.

2. **Signals interact in non-linear ways.** Whether a page needs refreshing depends on a combination of signals that shift depending on context: a short AI-generated page (`provider_used`, `word_count`) decays differently from a long human-written piece. A high `search_volume` page losing `ctr` is far more urgent than a low-volume page with the same loss. No if-statement can capture these joint patterns.

3. **Missingness is itself a signal.** `has_word_count = False` systematically identifies a specific `content_type` with its own decay dynamics. A rule cannot leverage a missingness pattern without becoming unmaintainable.

4. **The business goal is a ranked queue, not a bucket.** Even if a fixed rule flags the right pages, it produces an unordered list. ML outputs a continuous probability score that sorts all 30 000 pages from highest to lowest decline risk, directly optimising Precision@100.

The code below demonstrates this concretely: a simple `ctr` threshold rule misses a large fraction of genuinely declining pages that look fine on `ctr` alone.

In [5]:
import pandas as pd
import numpy as np
_CSV = r'C:/Users/Admin/Desktop/Krish/Projects/Flyrank ML/Machine Learning/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(_CSV)
df['is_declining_label'] = (df['trend_direction'] == 'down')

# Simple rule: flag pages in the bottom 10th percentile of ctr
threshold = df['ctr'].quantile(0.10)
rule_flags    = df['ctr'] <= threshold
true_decliners = df['is_declining_label']

rule_recall    = (rule_flags & true_decliners).sum() / true_decliners.sum()
rule_precision = (rule_flags & true_decliners).sum() / rule_flags.sum()

# Decliners the rule MISSES (above-threshold ctr but still declining)
missed = df[true_decliners & ~rule_flags]

print(f'CTR threshold used: {threshold:.4f} (10th percentile)')
print(f'Rule recall   (fraction of real decliners caught): {rule_recall:.2%}')
print(f'Rule precision (fraction of flagged that are real): {rule_precision:.2%}')
print(f'Decliners the rule misses entirely: {len(missed):,}')
print(f'Missed decliners -- avg ctr: {missed["ctr"].mean():.4f}, '
      f'avg avg_position: {missed["avg_position"].mean():.1f}')
print('\nConclusion: many genuine decliners look fine on ctr alone.')
print('An ML model combining ctr, engagement_rate, scroll_rate, word_count,')
print('has_flags, and content_age_days signals can find these missed cases.')

CTR threshold used: 0.0000 (10th percentile)
Rule recall   (fraction of real decliners caught): 40.35%
Rule precision (fraction of flagged that are real): 49.67%
Decliners the rule misses entirely: 9,700
Missed decliners -- avg ctr: 0.5434, avg avg_position: 14.1

Conclusion: many genuine decliners look fine on ctr alone.
An ML model combining ctr, engagement_rate, scroll_rate, word_count,
has_flags, and content_age_days signals can find these missed cases.


## Self-check

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere -- `content_id` and `client_id` are pseudonyms only
- [x] Claims use careful words: *observed*, *measured*, *directional*, *decision-support*
- [x] Committed to repo under `work/notebooks/`

---

**One-paragraph frame:**

> An editor uses this model's output to decide which specific content pages to prioritise for a refresh each week. The model scores every content item by its predicted probability of decline and ranks the full queue from highest to lowest risk, so the team works on the most urgent pages first. It is trained to predict `is_declining_label` (`trend_direction == 'down'`) -- an observed traffic-trend outcome derived from real impression and click data, not an arbitrary threshold rule -- and is evaluated by Precision@100: what fraction of the 100 highest-scored pages genuinely needed attention. A false negative (missing a real decliner) costs ongoing organic traffic loss that compounds over time; a false positive (flagging a healthy page) wastes irreplaceable editorial hours and risks disrupting a page that is already performing well. A plain if-statement rule cannot capture this because no single signal predicts decline in isolation -- it is the interaction between signals like `ctr`, `engagement_rate`, `scroll_rate`, `word_count`, `has_search_volume`, and `content_age_days` that reveals the underlying pattern.